<a href="https://colab.research.google.com/github/emanaak04-svg/medical-xai/blob/main/04_model_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transparent Medical Image Classification via Explainable AI
## Phase 2 — Model Architecture & Transfer Learning


**Objective:** This notebook defines the model architecture. Rather than training a CNN from scratch — which requires millions of images and weeks of compute — I apply transfer learning using ResNet-50 pretrained on ImageNet. The final classification layer is replaced and fine-tuned on chest X-ray data, a methodology validated by Rajpurkar et al. (2017) in the landmark CheXNet paper.

**Author:** Eman Ayman Ahmed Abukhousa  

**Program:** BSc Data Science & Artificial Intelligence, IITG — Year 3

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torchsummary import summary
import warnings
warnings.filterwarnings('ignore')

# Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment verified.")
print(f"Compute device      : {device}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")

Environment verified.
Compute device      : cuda
PyTorch version     : 2.10.0+cu128
CUDA available      : True
GPU                 : Tesla T4


## 01. Why ResNet-50?

ResNet-50 is a 50-layer deep convolutional neural network that introduced residual connections — shortcut paths that allow gradients to flow through the network without vanishing. This architectural innovation solved the degradation problem in very deep networks (He et al., 2016). For medical imaging, its depth allows it to learn hierarchical features from low-level edges to high-level pathological patterns. Critically, its pretrained ImageNet weights provide a strong initialization that dramatically reduces the data requirements for fine-tuning.

In [2]:
# ── Load pretrained ResNet-50 ──────────────────────────────────────
# ImageNet pretrained weights provide a strong feature extractor
# baseline. The network has already learned to detect edges, textures,
# and structural patterns — transferable to radiographic features.

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# ── Freeze all layers except the final block ───────────────────────
# Freezing early layers preserves low-level features learned from
# ImageNet. Only the final residual block and classifier are trained,
# reducing overfitting risk on the medical domain.

for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# ── Replace classification head ────────────────────────────────────
# Original ResNet-50 outputs 1000 ImageNet classes.
# We replace the final fully connected layer with a task-specific
# head for 15-class multilabel chest X-ray classification.

n_classes = 15
in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, n_classes)
)

model = model.to(device)

# Parameter audit
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params   = total_params - trainable_params

print("ResNet-50 architecture — transfer learning configuration")
print("─" * 55)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"Frozen parameters   : {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")
print("─" * 55)
print(f"Input shape         : (batch, 3, 224, 224)")
print(f"Output shape        : (batch, {n_classes}) — one logit per condition")
print(f"Classification head : Dropout(0.5) → Linear(2048→512) → ReLU → Dropout(0.3) → Linear(512→15)")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 140MB/s]


ResNet-50 architecture — transfer learning configuration
───────────────────────────────────────────────────────
Total parameters    : 24,564,815
Trainable parameters: 16,021,519 (65.2%)
Frozen parameters   : 8,543,296 (34.8%)
───────────────────────────────────────────────────────
Input shape         : (batch, 3, 224, 224)
Output shape        : (batch, 15) — one logit per condition
Classification head : Dropout(0.5) → Linear(2048→512) → ReLU → Dropout(0.3) → Linear(512→15)


## 02. Preprocessing Pipeline — PyTorch Implementation
The preprocessing strategy defined in notebook 02 is now implemented as a PyTorch transforms pipeline. Training and validation/test sets receive different transforms — augmentation is applied only during training.

In [3]:
# ── PyTorch transforms pipeline ────────────────────────────────────
# Implements the preprocessing specification frozen in notebook 02.
# Training transforms include augmentation — validation and test
# transforms apply normalization only, ensuring unbiased evaluation.

# Chest X-ray specific normalization parameters (Rajpurkar et al., 2017)
MEAN = 0.5330
STD  = 0.0349

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN, MEAN, MEAN],
                         std=[STD,  STD,  STD]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[MEAN, MEAN, MEAN],
                         std=[STD,  STD,  STD]),
])

# ── Verify pipeline with synthetic tensor ──────────────────────────
# Confirm the model accepts correctly shaped input before real data
test_input = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    test_output = model(test_input)

print("Transform pipeline verified")
print("─" * 45)
print(f"Train transforms    : {len(train_transforms.transforms)} steps")
print(f"Val transforms      : {len(val_transforms.transforms)} steps")
print(f"Test input shape    : {list(test_input.shape)}")
print(f"Model output shape  : {list(test_output.shape)}")
print(f"Output per sample   : {test_output.shape[1]} logits (one per condition)")
print("─" * 45)
print("\nModel accepts input correctly — ready for training.")

Transform pipeline verified
─────────────────────────────────────────────
Train transforms    : 8 steps
Val transforms      : 4 steps
Test input shape    : [4, 3, 224, 224]
Model output shape  : [4, 15]
Output per sample   : 15 logits (one per condition)
─────────────────────────────────────────────

Model accepts input correctly — ready for training.


## 03. Loss Function & Optimizer Configuration
Weighted cross-entropy loss is applied using the class weights computed in notebook 03. The Adam optimizer is used with a conservative learning rate — too high a learning rate risks destroying the pretrained ImageNet weights during fine-tuning.

In [6]:
# ── Loss function — weighted cross entropy ─────────────────────────
# Class weights from notebook 03 — inversely proportional to
# class frequency. Rare conditions receive higher loss penalties.

real_distribution = {
    'No Finding'        : 60361, 'Infiltration'      : 9547,
    'Effusion'          : 13317, 'Atelectasis'       : 11559,
    'Nodule'            : 6331,  'Mass'              : 5782,
    'Pneumonia'         : 1431,  'Pleural_Thickening': 3385,
    'Cardiomegaly'      : 2776,  'Emphysema'         : 2516,
    'Edema'             : 2303,  'Consolidation'     : 4667,
    'Pneumothorax'      : 5302,  'Fibrosis'          : 1686,
    'Hernia'            : 227,
}

total      = sum(real_distribution.values())
n_classes  = len(real_distribution)
conditions = list(real_distribution.keys())

weights = torch.tensor([
    total / (n_classes * count)
    for count in real_distribution.values()
], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

# ── Optimizer — Adam with fine-tuning learning rate ────────────────
# lr=1e-4 is deliberately conservative — aggressive learning rates
# risk catastrophic forgetting of pretrained ImageNet features.
# Layer4 and fc head use a slightly higher lr for faster adaptation.

optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3},
], weight_decay=1e-4)

# ── Learning rate scheduler ────────────────────────────────────────
# Reduce LR by 50% if validation loss plateaus for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

print("Training configuration")
print("─" * 45)
print(f"Loss function       : Weighted CrossEntropyLoss")
print(f"Optimizer           : Adam (weight decay=1e-4)")
print(f"Layer4 LR           : 1e-4")
print(f"Classifier head LR  : 1e-3")
print(f"LR scheduler        : ReduceLROnPlateau (patience=3)")
print(f"Class weights range : {weights.min().item():.3f} → {weights.max().item():.3f}")
print("─" * 45)
print("\nTraining configuration locked.")

Training configuration
─────────────────────────────────────────────
Loss function       : Weighted CrossEntropyLoss
Optimizer           : Adam (weight decay=1e-4)
Layer4 LR           : 1e-4
Classifier head LR  : 1e-3
LR scheduler        : ReduceLROnPlateau (patience=3)
Class weights range : 0.145 → 38.529
─────────────────────────────────────────────

Training configuration locked.
